<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/11_iac.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-04-ml-architect/11_iac.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# 11 — Infrastructure as Code for ML Platforms

**Objectives**
- Generate Terraform and Kubernetes manifests programmatically.
- Enforce policy-as-code checks before anything reaches a cluster.
- Treat infrastructure changes with the same review rigor as application code.


In [ ]:
config = {
    "service": "customer-support",
    "image": "ghcr.io/example/customer-support:1.0.0",
    "replicas": 2,
    "cpu": "500m",
    "memory": "1Gi",
    "port": 8000,
}
print(config)

In [ ]:
tf = f"""
resource "kubernetes_namespace" "{config['service'].replace('-', '_')}" {{
  metadata {{
    name = "{config['service']}"
  }}
}}

resource "helm_release" "{config['service'].replace('-', '_')}" {{
  name      = "{config['service']}"
  chart     = "./helm/{config['service']}"
  namespace = kubernetes_namespace.{config['service'].replace('-', '_')}.metadata[0].name
}}
"""
Path = __import__("pathlib").Path
Path("/tmp/iac_demo").mkdir(exist_ok=True)
Path("/tmp/iac_demo/main.tf").write_text(tf)
print("terraform_generated: /tmp/iac_demo/main.tf")
print(tf)

In [ ]:
import yaml

manifest = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": config["service"]},
    "spec": {
        "replicas": config["replicas"],
        "selector": {"matchLabels": {"app": config["service"]}},
        "template": {
            "metadata": {"labels": {"app": config["service"]}},
            "spec": {
                "containers": [
                    {
                        "name": "api",
                        "image": config["image"],
                        "ports": [{"containerPort": config["port"]}],
                        "resources": {
                            "requests": {"cpu": config["cpu"], "memory": config["memory"]},
                            "limits": {"cpu": config["cpu"], "memory": config["memory"]},
                        },
                        "securityContext": {
                            "runAsNonRoot": True,
                            "allowPrivilegeEscalation": False,
                        },
                    }
                ]
            },
        },
    },
}
Path("/tmp/iac_demo/deployment.yaml").write_text(yaml.safe_dump(manifest))
print(yaml.safe_dump(manifest))

In [ ]:
# Policy-as-code: catch the classic production foot-guns before apply
def check_policies(manifest: dict) -> list[str]:
    container = manifest["spec"]["template"]["spec"]["containers"][0]
    violations = []
    if container["image"].endswith(":latest"):
        violations.append("image tag must be immutable (not :latest)")
    if "limits" not in container.get("resources", {}):
        violations.append("resource limits required")
    if not container.get("securityContext", {}).get("runAsNonRoot"):
        violations.append("runAsNonRoot required")
    return violations


violations = check_policies(manifest)
print("policy_violations:", violations)

**Review checklist**
1. Images pinned by digest or immutable tag.
2. Resource requests *and* limits set — requests schedule, limits protect neighbors.
3. Security context explicit; no privilege escalation.
4. Changes go through plan → review → apply, never `apply -auto-approve` in prod.
